<a href="https://colab.research.google.com/github/Mohana-Siva/Sudoku-Puzzle/blob/main/distributed_sudoku_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Distributed Sudoku-Solving Agents
### Agentic AI project — comparing Centralized, Hierarchical, Decentralized, and Sequential/Pipeline coordination

**Idea:** a 9x9 Sudoku is split into 9 boxes (3x3 sub-grids). Each box is owned by one agent.
The puzzle-solving logic (constraint propagation + backtracking) is identical across all four
architectures, so the *only* variable is **how the agents are organized and communicate**. An LLM
(Groq, openai/gpt) is used for the genuinely "agentic" layer: conflict
resolution reasoning, negotiation between peers, and handoff notes. The deterministic core
guarantees the notebook always finishes with a valid, complete board, LLM or not.

Run the cells top to bottom. The last section runs all four architectures on the same puzzle and
compares them (messages sent, rounds, LLM calls, time) — useful straight in your report.


## 1. Setup

In [ ]:
!pip install -q groq

import json, time, random
from typing import Dict, Tuple, List

USE_LLM = True  # flip to False to run fully offline
client = None
if USE_LLM:
    from groq import Groq
    API_KEY = None
    try:
        from google.colab import userdata
        API_KEY = userdata.get('GROQ_API_KEY')
    except Exception:
        pass
    if API_KEY:
        client = Groq(api_key=API_KEY)
    else:
        print("No GROQ_API_KEY secret found. Running with offline fallback")
        USE_LLM = False

MODEL = "openai/gpt-oss-120b"  #model-used

def llm_reason(system_prompt: str, user_prompt: str, metrics=None, retries=3) -> str:
    """Ask Groq for a short piece of agent reasoning / communication.
    Falls back to a canned string if no API key is configured, or if the free
    tier's rate limit can't be cleared after a few backed-off retries, so the
    notebook always finishes running instead of crashing mid-cell."""
    if not USE_LLM or client is None:
        return "[offline] Proceeding based on local constraint analysis."

    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                max_tokens=150,
            )
            if metrics:
                metrics.log_llm_call()
            time.sleep(2.5)  # free tier's 30 requests/minute cap
            return resp.choices[0].message.content.strip()
        except Exception as e:
            is_rate_limit = "429" in str(e) or "rate_limit" in str(e).lower()
            if is_rate_limit and attempt < retries - 1:
                wait = 15 * (attempt + 1)  # 15s, 30s, 45s
                print(f"  (rate limited, waiting {wait}s and retrying...)")
                time.sleep(wait)
                continue
            print(f"  (LLM call failed after retries: {e}. Falling back to offline text.)")
            return "[offline - rate limited] Proceeding based on local constraint analysis."

In [ ]:
# Diagnostic code
print("USE_LLM:", USE_LLM)
print("client configured:", client is not None)
if USE_LLM:
    test = llm_reason("You are a test assistant.", "Reply with exactly: LLM is working.")
    print("Test call result:", test)


USE_LLM: True
client configured: True
Test call result: LLM is working.


## 2. Sudoku board and puzzle

In [ ]:
class SudokuBoard:
    def __init__(self, grid):
        self.grid = [row[:] for row in grid]

    def clone(self):
        return SudokuBoard(self.grid)

    def get_row(self, r): return self.grid[r][:]
    def get_col(self, c): return [self.grid[r][c] for r in range(9)]

    def get_box_cells(self, b):
        br, bc = (b // 3) * 3, (b % 3) * 3
        return [(r, c) for r in range(br, br + 3) for c in range(bc, bc + 3)]

    def get_box_values(self, b):
        return [self.grid[r][c] for r, c in self.get_box_cells(b)]

    def candidates(self, r, c):
        if self.grid[r][c] != 0:
            return set()
        used = set(self.get_row(r)) | set(self.get_col(c))
        b = (r // 3) * 3 + (c // 3)
        used |= set(self.get_box_values(b))
        return set(range(1, 10)) - used

    def is_valid_placement(self, r, c, v):
        return v in self.candidates(r, c)

    def place(self, r, c, v):
        if not self.is_valid_placement(r, c, v):
            return False
        self.grid[r][c] = v
        return True

    def is_complete(self):
        return all(cell != 0 for row in self.grid for cell in row)

    def empty_count(self):
        return sum(row.count(0) for row in self.grid)

    def pretty(self):
        lines = []
        for i, row in enumerate(self.grid):
            if i % 3 == 0 and i:
                lines.append("-" * 21)
            line = ""
            for j, v in enumerate(row):
                if j % 3 == 0 and j:
                    line += "| "
                line += (str(v) if v else ".") + " "
            lines.append(line)
        return "\n".join(lines)


SAMPLE_PUZZLE = [
    [5,3,0, 0,7,0, 0,0,0],
    [6,0,0, 1,9,5, 0,0,0],
    [0,9,8, 0,0,0, 0,6,0],

    [8,0,0, 0,6,0, 0,0,3],
    [4,0,0, 8,0,3, 0,0,1],
    [7,0,0, 0,2,0, 0,0,6],

    [0,6,0, 0,0,0, 2,8,0],
    [0,0,0, 4,1,9, 0,0,5],
    [0,0,0, 0,8,0, 0,7,9],
]

print(SudokuBoard(SAMPLE_PUZZLE).pretty())


5 3 . | . 7 . | . . . 
6 . . | 1 9 5 | . . . 
. 9 8 | . . . | . 6 . 
---------------------
8 . . | . 6 . | . . 3 
4 . . | 8 . 3 | . . 1 
7 . . | . 2 . | . . 6 
---------------------
. 6 . | . . . | 2 8 . 
. . . | 4 1 9 | . . 5 
. . . | . 8 . | . 7 9 


## 3.Backtracking fallback

In [ ]:
class Metrics:
    """Tracks coordination cost so the four architectures can be compared fairly."""
    def __init__(self, name):
        self.name = name
        self.messages = 0
        self.rounds = 0
        self.llm_calls = 0
        self.start = time.time()

    def log_message(self, n=1): self.messages += n
    def log_round(self): self.rounds += 1
    def log_llm_call(self): self.llm_calls += 1
    def elapsed(self): return round(time.time() - self.start, 3)

    def summary(self):
        return {
            "architecture": self.name,
            "messages": self.messages,
            "rounds": self.rounds,
            "llm_calls": self.llm_calls,
            "time_sec": self.elapsed(),
        }


def solve_box_naked_singles(board: SudokuBoard, box_id: int) -> Dict[Tuple[int, int], int]:
    """A single agent's local reasoning: for each empty cell in ITS box, if
    exactly one candidate value survives (given the board it currently sees),
    propose that value. This is deliberately simple/local -- it's the
    coordination pattern around it that differs per architecture."""
    proposals = {}
    for (r, c) in board.get_box_cells(box_id):
        if board.grid[r][c] == 0:
            cands = board.candidates(r, c)
            if len(cands) == 1:
                proposals[(r, c)] = next(iter(cands))
    return proposals


def backtrack_solve(board: SudokuBoard) -> bool:
    """Escalation fallback: if local reasoning alone stalls, finish the puzzle
    with brute-force search so every architecture is guaranteed to converge."""
    for r in range(9):
        for c in range(9):
            if board.grid[r][c] == 0:
                for v in sorted(board.candidates(r, c)):
                    board.grid[r][c] = v
                    if backtrack_solve(board):
                        return True
                    board.grid[r][c] = 0
                return False
    return True


## 4. Architecture 1 — Centralized

One controller agent has full visibility of the board. Each round it dispatches a "solve your
box" task to all 9 worker agents, collects their proposals, detects conflicts itself, and (via the
LLM) decides how to resolve them before applying anything. **All communication passes through the
controller** — workers never talk to each other.

In [ ]:
class CentralizedController:
    def __init__(self, board: SudokuBoard, max_rounds=30):
        self.board = board
        self.max_rounds = max_rounds
        self.metrics = Metrics("Centralized")

    def run(self, verbose=True):
        for rnd in range(self.max_rounds):
            self.metrics.log_round()
            all_proposals, conflicts = {}, []
            for box_id in range(9):
                self.metrics.log_message()  # controller -> worker: "solve your box"
                proposal = solve_box_naked_singles(self.board, box_id)
                self.metrics.log_message()  # worker -> controller: result
                for cell, val in proposal.items():
                    if cell in all_proposals and all_proposals[cell] != val:
                        conflicts.append((cell, all_proposals[cell], val))
                    else:
                        all_proposals[cell] = val

            if conflicts:
                note = llm_reason(
                    "You are a central controller resolving Sudoku worker-agent conflicts.",
                    f"Conflicting cell proposals (cell, val_a, val_b): {conflicts}. "
                    "In one sentence, say how you'll decide which value is Sudoku-valid.",
                    self.metrics,
                )
                if verbose: print(f"[Round {rnd}] Controller (conflict): {note}")

            applied = sum(self.board.place(r, c, v) for (r, c), v in all_proposals.items())

            # The controller always reports back to itself/the log each round --
            # this is the visible "agentic" coordination step, separate from the
            # rare conflict-resolution case above. Capped so cost stays bounded.
            if rnd < 3:
                note = llm_reason(
                    "You are a central controller coordinating 9 Sudoku worker agents.",
                    f"This round you applied {applied} cell(s) across the board; "
                    f"{self.board.empty_count()} cells remain empty. In one sentence, "
                    "summarize progress and what you'll direct the workers to do next round.",
                    self.metrics,
                )
                if verbose: print(f"[Round {rnd}] Controller: {note}")

            if verbose:
                print(f"Round {rnd}: applied {applied}, {self.board.empty_count()} empty left")
            if applied == 0:
                break
            if self.board.is_complete():
                break
        return self.board, self.metrics


board = SudokuBoard(SAMPLE_PUZZLE)
ctrl = CentralizedController(board)
final_board, metrics = ctrl.run()
print("\n" + final_board.pretty())
print(metrics.summary())


[Round 0] Controller: We've placed four numbers, leaving 47 cells empty, so I'll now have the workers each target a distinct unsolved region—rows, columns, or boxes—to hunt for hidden singles, naked pairs, and pointing/claiming eliminations in the next pass.
Round 0: applied 4, 47 empty left
[Round 1] Controller: We’ve filled five new numbers, leaving 42 blanks, so I’ll have the workers focus next on scanning each row, column and box for hidden singles and pointing pairs to lock in additional placements.
Round 1: applied 5, 42 empty left
[Round 2] Controller: We placed 7 cells, reducing the board to 35 empties, and now I’ll have the workers each scan their assigned rows, columns, and boxes for hidden‑single opportunities and update all
Round 2: applied 7, 35 empty left
Round 3: applied 6, 29 empty left
Round 4: applied 5, 24 empty left
Round 5: applied 3, 21 empty left
Round 6: applied 2, 19 empty left
Round 7: applied 7, 12 empty left
Round 8: applied 10, 2 empty left
Round 9: applied

## 5. Architecture 2 — Hierarchical

A supervisor oversees 3 band managers (top/middle/bottom band of boxes); each manager oversees 3
box agents. Box agents report to their manager, managers resolve conflicts *within* their band and
report a merged result up; the supervisor only steps in for conflicts that cross bands.

In [ ]:
class BandManager:
    def __init__(self, band_id, box_ids, metrics):
        self.band_id = band_id
        self.box_ids = box_ids
        self.metrics = metrics

    def collect(self, board):
        proposals = {}
        for box_id in self.box_ids:
            self.metrics.log_message()  # manager -> box agent
            p = solve_box_naked_singles(board, box_id)
            self.metrics.log_message()  # box agent -> manager
            proposals.update(p)
        return proposals


class HierarchicalController:
    def __init__(self, board, max_rounds=30):
        self.board = board
        self.max_rounds = max_rounds
        self.metrics = Metrics("Hierarchical")
        self.bands = [BandManager(i, [i*3, i*3+1, i*3+2], self.metrics) for i in range(3)]

    def run(self, verbose=True):
        for rnd in range(self.max_rounds):
            self.metrics.log_round()
            band_results = []
            for band in self.bands:
                proposals = band.collect(self.board)
                self.metrics.log_message()  # band manager -> supervisor
                band_results.append(proposals)

            merged, conflicts = {}, []
            for proposals in band_results:
                for cell, val in proposals.items():
                    if cell in merged and merged[cell] != val:
                        conflicts.append((cell, merged[cell], val))
                    else:
                        merged[cell] = val

            if conflicts:
                note = llm_reason(
                    "You are a supervisor agent resolving cross-band Sudoku conflicts.",
                    f"Conflicts across bands: {conflicts}. In one sentence, say how you'll pick the valid value.",
                    self.metrics,
                )
                if verbose: print(f"[Round {rnd}] Supervisor (conflict): {note}")

            applied = sum(self.board.place(r, c, v) for (r, c), v in merged.items())

            # Supervisor's regular round-summary check-in with the band managers --
            # this is the visible coordination step, independent of whether a
            # cross-band conflict happened to occur this round.
            if rnd < 3:
                note = llm_reason(
                    "You are a supervisor agent overseeing 3 band managers, each running 3 Sudoku box agents.",
                    f"This round the bands together applied {applied} cell(s); "
                    f"{self.board.empty_count()} cells remain empty. In one sentence, "
                    "summarize progress across the bands and what to focus on next round.",
                    self.metrics,
                )
                if verbose: print(f"[Round {rnd}] Supervisor: {note}")

            if verbose:
                print(f"Round {rnd}: applied {applied}, {self.board.empty_count()} empty left")
            if applied == 0:
                break
            if self.board.is_complete():
                break
        return self.board, self.metrics


board = SudokuBoard(SAMPLE_PUZZLE)
ctrl = HierarchicalController(board)
final_board, metrics = ctrl.run()
print("\n" + final_board.pretty())
print(metrics.summary())


[Round 0] Supervisor: The bands have collectively filled only 4 cells, leaving 47 empties, so the next round should concentrate on identifying and resolving high‑constraint rows, columns, or boxes (e.g., hidden singles or naked pairs) to accelerate overall reduction.
Round 0: applied 4, 47 empty left
[Round 1] Supervisor: 
Round 1: applied 5, 42 empty left
[Round 2] Supervisor: The bands collectively filled 7 cells, leaving 35 empty, so focus the next round on the remaining high‑constraint cells to maximize eliminations and advance toward completion.
Round 2: applied 7, 35 empty left
Round 3: applied 6, 29 empty left
Round 4: applied 5, 24 empty left
Round 5: applied 3, 21 empty left
Round 6: applied 2, 19 empty left
Round 7: applied 7, 12 empty left
Round 8: applied 10, 2 empty left
Round 9: applied 2, 0 empty left

5 3 4 | 6 7 8 | 9 1 2 
6 7 2 | 1 9 5 | 3 4 8 
1 9 8 | 3 4 2 | 5 6 7 
---------------------
8 5 9 | 7 6 1 | 4 2 3 
4 2 6 | 8 5 3 | 7 9 1 
7 1 3 | 9 2 4 | 8 5 6 
-----------

## 6. Architecture 3 — Decentralized

No central authority at all. Each box agent only knows about **peer** agents that share a row-band
or column-band with it (its natural constraint neighbors), and checks proposals against them
directly, peer-to-peer, in randomized order each round. If two peers move in the same round and
clash, the loser just retries next round -- there's no one to escalate to.

In [ ]:
def box_neighbors(box_id):
    row_band, col_band = box_id // 3, box_id % 3
    return {b for b in range(9) if b != box_id and (b // 3 == row_band or b % 3 == col_band)}


class DecentralizedAgent:
    def __init__(self, box_id, metrics):
        self.box_id = box_id
        self.metrics = metrics
        self.neighbors = box_neighbors(box_id)

    def negotiate_and_propose(self, board):
        proposal = solve_box_naked_singles(board, self.box_id)
        accepted = {}
        for cell, val in proposal.items():
            r, c = cell
            still_valid = True
            for _ in self.neighbors:
                self.metrics.log_message()  # peer-to-peer check-in
                if not board.is_valid_placement(r, c, val):
                    still_valid = False
            if still_valid:
                accepted[cell] = val
        return accepted


class DecentralizedController:
    def __init__(self, board, max_rounds=30, seed=42):
        self.board = board
        self.max_rounds = max_rounds
        self.metrics = Metrics("Decentralized")
        self.agents = {i: DecentralizedAgent(i, self.metrics) for i in range(9)}
        self.rng = random.Random(seed)

    def run(self, verbose=True):
        for rnd in range(self.max_rounds):
            self.metrics.log_round()
            order = list(range(9))
            self.rng.shuffle(order)
            applied, clashes = 0, []
            for box_id in order:
                proposal = self.agents[box_id].negotiate_and_propose(self.board)
                for (r, c), v in proposal.items():
                    if self.board.grid[r][c] == 0:
                        if self.board.place(r, c, v):
                            applied += 1
                        else:
                            clashes.append((box_id, r, c, v))

            if clashes and self.metrics.llm_calls < 5:  # keep API call volume bounded
                note = llm_reason(
                    "You are one of several peer Sudoku agents with no central coordinator.",
                    f"These proposals clashed because another peer moved first this round: {clashes}. "
                    "In one sentence, say what a peer should re-check before retrying.",
                    self.metrics,
                )
                if verbose: print(f"[Round {rnd}] Peer note (clash): {note}")

            # Even with no clashes, peers still "check in" with each other each
            # round -- that ongoing chatter is the visible decentralized
            # coordination step. Capped so cost stays bounded.
            if rnd < 3:
                note = llm_reason(
                    "You are one of several peer Sudoku agents with no central coordinator, "
                    "checking in with your neighbors after a round of local proposals.",
                    f"This round the group applied {applied} cell(s) with {len(clashes)} clash(es); "
                    f"{self.board.empty_count()} cells remain empty. In one sentence, note how "
                    "the peer group should proceed next round.",
                    self.metrics,
                )
                if verbose: print(f"[Round {rnd}] Peer group: {note}")

            if verbose:
                print(f"Round {rnd}: applied {applied}, {self.board.empty_count()} empty left")
            if applied == 0:
                break
            if self.board.is_complete():
                break
        return self.board, self.metrics


board = SudokuBoard(SAMPLE_PUZZLE)
ctrl = DecentralizedController(board)
final_board, metrics = ctrl.run()
print("\n" + final_board.pretty())
print(metrics.summary())


[Round 0] Peer group: The peers should continue the next round by re‑evaluating all rows, columns, and boxes to find any new hidden‑single or naked‑pair opportunities and propagate those deductions.
Round 0: applied 4, 47 empty left
[Round 1] Peer group: Proceed by exchanging the updated board, then collectively target the remaining 40 empty cells using deterministic strategies (e.g., naked/hidden singles, pointing pairs) before resorting to any further speculative placements.
Round 1: applied 7, 40 empty left
[Round 2] Peer group: Proceed by each peer selecting the next highest‑impact empty cells (e.g., those with the fewest candidate numbers), proposing consistent values, sharing those proposals, and iteratively reconciling any conflicts before committing further placements.
Round 2: applied 9, 31 empty left
Round 3: applied 8, 23 empty left
Round 4: applied 5, 18 empty left
Round 5: applied 12, 6 empty left
Round 6: applied 6, 0 empty left

5 3 4 | 6 7 8 | 9 1 2 
6 7 2 | 1 9 5 | 3 4

## 7. Architecture 4 — Sequential / Pipeline

Box agents run in a fixed order like an assembly line. Each agent only ever sees the board *as
handed to it by the previous stage*, solves what it can, and passes the updated board downstream.
There's no conflict detection needed by construction -- later stages simply build on earlier,
final decisions.

In [ ]:
class PipelineController:
    def __init__(self, board, order=None, max_rounds=30):
        self.board = board
        self.order = order or list(range(9))
        self.max_rounds = max_rounds
        self.metrics = Metrics("Pipeline")

    def run(self, verbose=True):
        for rnd in range(self.max_rounds):
            self.metrics.log_round()
            applied = 0
            for i, box_id in enumerate(self.order):
                proposal = solve_box_naked_singles(self.board, box_id)
                stage_applied = sum(self.board.place(r, c, v) for (r, c), v in proposal.items())
                applied += stage_applied
                if stage_applied and i < len(self.order) - 1:
                    self.metrics.log_message()  # handoff to next stage in the line

            # Each pass down the line ends with a handoff note to whoever runs the
            # next pass -- the visible coordination step for a pipeline. Capped
            # so cost stays bounded.
            if rnd < 3:
                note = llm_reason(
                    "You are the final stage of one pass through a sequential Sudoku-solving pipeline.",
                    f"This pass resolved {applied} cell(s) total; {self.board.empty_count()} cells remain empty. "
                    "Give a one-sentence handoff note for the agent starting the next pass.",
                    self.metrics,
                )
                if verbose: print(f"[Round {rnd}] Handoff note: {note}")

            if verbose:
                print(f"Round {rnd}: applied {applied}, {self.board.empty_count()} empty left")
            if applied == 0:
                break
            if self.board.is_complete():
                break
        return self.board, self.metrics


board = SudokuBoard(SAMPLE_PUZZLE)
ctrl = PipelineController(board)
final_board, metrics = ctrl.run()
print("\n" + final_board.pretty())
print(metrics.summary())


[Round 0] Handoff note: We resolved five cells (now 46 empty), so the next pass should prioritize hidden singles in rows 3, 5, 7 and candidate elimination in column 2 to unlock further progress.
Round 0: applied 5, 46 empty left
[Round 1] Handoff note: We've filled 7 cells, leaving 39 blanks, so the
Round 1: applied 7, 39 empty left
[Round 2] Handoff note: We resolved 11 cells and now have 28 blanks remaining, so
Round 2: applied 11, 28 empty left
Round 3: applied 9, 19 empty left
Round 4: applied 9, 10 empty left
Round 5: applied 10, 0 empty left

5 3 4 | 6 7 8 | 9 1 2 
6 7 2 | 1 9 5 | 3 4 8 
1 9 8 | 3 4 2 | 5 6 7 
---------------------
8 5 9 | 7 6 1 | 4 2 3 
4 2 6 | 8 5 3 | 7 9 1 
7 1 3 | 9 2 4 | 8 5 6 
---------------------
9 6 1 | 5 3 7 | 2 8 4 
2 8 7 | 4 1 9 | 6 3 5 
3 4 5 | 2 8 6 | 1 7 9 
{'architecture': 'Pipeline', 'messages': 32, 'rounds': 6, 'llm_calls': 3, 'time_sec': 9.665}


## 8. Head-to-head comparison

Runs all four architectures on the *same* starting puzzle and reports messages exchanged, rounds,
LLM calls, wall-clock time, and whether brute-force escalation was needed to finish. This table is
the core evidence for your report's "compare the architectures" section.

In [ ]:
import pandas as pd

architectures = [CentralizedController, HierarchicalController, DecentralizedController, PipelineController]
results = []

for Controller in architectures:
    board = SudokuBoard(SAMPLE_PUZZLE)
    ctrl = Controller(board)
    final_board, metrics = ctrl.run(verbose=False)

    escalated = False
    if not final_board.is_complete():
        escalated = True
        backtrack_solve(final_board)

    summary = metrics.summary()
    summary["escalated_to_backtracking"] = escalated
    summary["solved"] = final_board.is_complete()
    results.append(summary)

    print(f"=== {summary['architecture']} ===")
    print(final_board.pretty())
    print(summary, "\n")

df = pd.DataFrame(results)
df


=== Centralized ===
5 3 4 | 6 7 8 | 9 1 2 
6 7 2 | 1 9 5 | 3 4 8 
1 9 8 | 3 4 2 | 5 6 7 
---------------------
8 5 9 | 7 6 1 | 4 2 3 
4 2 6 | 8 5 3 | 7 9 1 
7 1 3 | 9 2 4 | 8 5 6 
---------------------
9 6 1 | 5 3 7 | 2 8 4 
2 8 7 | 4 1 9 | 6 3 5 
3 4 5 | 2 8 6 | 1 7 9 
{'architecture': 'Centralized', 'messages': 180, 'rounds': 10, 'llm_calls': 3, 'time_sec': 9.468, 'escalated_to_backtracking': False, 'solved': True} 

=== Hierarchical ===
5 3 4 | 6 7 8 | 9 1 2 
6 7 2 | 1 9 5 | 3 4 8 
1 9 8 | 3 4 2 | 5 6 7 
---------------------
8 5 9 | 7 6 1 | 4 2 3 
4 2 6 | 8 5 3 | 7 9 1 
7 1 3 | 9 2 4 | 8 5 6 
---------------------
9 6 1 | 5 3 7 | 2 8 4 
2 8 7 | 4 1 9 | 6 3 5 
3 4 5 | 2 8 6 | 1 7 9 
{'architecture': 'Hierarchical', 'messages': 210, 'rounds': 10, 'llm_calls': 3, 'time_sec': 9.271, 'escalated_to_backtracking': False, 'solved': True} 

=== Decentralized ===
5 3 4 | 6 7 8 | 9 1 2 
6 7 2 | 1 9 5 | 3 4 8 
1 9 8 | 3 4 2 | 5 6 7 
---------------------
8 5 9 | 7 6 1 | 4 2 3 
4 2 6 | 8 5 3 | 

,architecture,messages,rounds,llm_calls,time_sec,escalated_to_backtracking,solved
0,Centralized,180,10,3,9.468,False,True
1,Hierarchical,210,10,3,9.271,False,True
2,Decentralized,204,7,3,9.635,False,True
3,Pipeline,32,6,3,9.627,False,True
